# Phase 5 step 1 — SAM 2 box refinement (T4, no training)

SAM 2 (Meta's "cut out the object in this box" model) redraws every mask of our best model `x101_2048`
(val PQ 0.4264, LB 0.37). Nothing is trained: the Mask R-CNN predictions are already in the repo
(`Result Phase 4/`), so this notebook only runs SAM 2 on them. Each mask gets a square crop around its box and
two prompts: the box alone (`box`), and the box plus 3 points along the filament (`boxpts`).

| Step | What | Time on T4 (estimate) |
|---|---|---|
| 0 | probe: 3 val images, prints seconds per mask | ~3 min (incl. model download) |
| 1 | val: ~1,200 masks of 141 images | ~15–30 min |
| 2 | test: ~1,600 masks of 180 images | ~20–40 min |

**Runtime:** T4 GPU (Runtime → Change runtime type). ~1 compute unit per hour. Close other Colab sessions first.

**How to run:** Runtime → Run all. Every step stops the notebook if a command fails. Results go to
`MyDrive/solar_filament/Phase 5/`; scoring (which prompt and combination helps) is done on the local computer.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
PHASE5_DIR = "/content/drive/MyDrive/solar_filament/Phase 5"   # SAM masks to send back
os.makedirs(PHASE5_DIR, exist_ok=True)
print("Drive ready:", PHASE5_DIR)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install SAM 2 (~2 min, no Detectron2 needed) and run the quick tests
`SAM2_BUILD_CUDA=0` skips an optional GPU extension that image prediction does not use; a warning about a missing
`_C` module later on is expected and harmless.

In [ ]:
!pip install -q -r requirements.txt
!SAM2_BUILD_CUDA=0 pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/sam2.git'
import torch, sam2
print("torch", torch.__version__, "| sam2 OK | GPU", torch.cuda.get_device_name(0))

!python -m pytest -q -p no:warnings tests/test_sam_refine.py
assert _exit_code == 0, "Tests failed - stop and tell Claude"

## 4. Helper

In [ ]:
import time

NAME = "x101_2048"

def run(args):
    """Run one sam_refine.py command; stop the notebook if it fails."""
    t0 = time.time()
    get_ipython().system(f"python src/sam_refine.py {args}")
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {args}")
    print(f"--- done in {(time.time() - t0) / 60:.1f} min")

## Step 0 — Probe (3 val images, ~3 min)
Downloads SAM 2.1 (large, ~900 MB) and refines the masks of 3 images. Look for the line
`[SAM] val: 3 images, ... s/mask`: ~1,200 val masks × this = time of step 1. The probe files are not kept.

In [ ]:
run(f'dump --raw "Result Phase 4/raw_preds_{NAME}_val.json.gz" --split val --limit 3 --out-dir /content/probe')

## Step 1 — Val (141 images)

In [ ]:
run(f'dump --raw "Result Phase 4/raw_preds_{NAME}_val.json.gz" --split val --out-dir "{PHASE5_DIR}"')

## Step 2 — Test (180 images)

In [ ]:
run(f'dump --raw "Result Phase 4/raw_preds_{NAME}_test.json.gz" --split test --out-dir "{PHASE5_DIR}"')

## 5. Send results back
Download the 4 files (also in `MyDrive/solar_filament/Phase 5/`) into the local folder `Result Phase 5/`, then tell
Claude. Please also copy the `[SAM] val: ... s/mask` and `Peak GPU memory` lines.
Then **disconnect the runtime** (Runtime → Disconnect and delete runtime) so it stops using compute units.

In [ ]:
from google.colab import files
for prompt in ("box", "boxpts"):
    for split in ("val", "test"):
        f = f"{PHASE5_DIR}/sam_{prompt}_{NAME}_{split}.json.gz"
        if os.path.exists(f):
            files.download(f)
        else:
            print("not found (step not finished?):", f)